## 1. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 30)

## 2. Load Data

In [3]:
# Load 2009-2010 data
year09_10 = pd.read_excel('C:/Projects/online-retail-rfm-segmentation/data/online_retail_II.xlsx', sheet_name='Year 2009-2010')

print("Shape", year09_10.shape)
year09_10.head(3)

Shape (525461, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


In [4]:
# Load 2010-2011 data
year10_11 = pd.read_excel('C:/Projects/online-retail-rfm-segmentation/data/online_retail_II.xlsx', sheet_name='Year 2010-2011')

print("Shape", year10_11.shape)
year10_11.head(3)

Shape (541910, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


## 3. Combine Dataframes

In [7]:
df = pd.concat([year09_10, year10_11], ignore_index=True)

print(df)

        Invoice StockCode                          Description  Quantity  \
0        489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1        489434    79323P                   PINK CHERRY LIGHTS        12   
2        489434    79323W                  WHITE CHERRY LIGHTS        12   
3        489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4        489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...         ...       ...                                  ...       ...   
1067366  581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
1067368  581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
1067369  581587     22138        BAKING SET 9 PIECE RETROSPOT          3   
1067370  581587      POST                              POSTAGE         1   

                InvoiceDate  Price  Customer ID         Country  
0       2009-12-01 07

## 3. Initial Data Quality Check

In [16]:
print(df.dtypes)
print()
print("Nulls per column: ", df.isnull().sum())
print()
duplicate_count = df.duplicated().sum()  #Count duplicates across all columns
print("Full row duplicates ALL: ", duplicate_count)

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Nulls per column:  Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Full row duplicates:  34335

Full row duplicates ALL:  34335


In [ ]:
# Convert Invoice to str
df["Invoice"] = df["Invoice"].astype(str)

## 4. Create Duplicates Dataframe and Remove from df

In [22]:
## Copy the duplicate rows to a separate dataframe
duplicates_df = df[df.duplicated()]
duplicates_df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom


In [21]:
print(duplicates_df.shape)

(34335, 8)


In [25]:
# Remove dupes
print("Rows Before: ", df.shape[0])

df = df.drop_duplicates().reset_index(drop=True)

print("Rows After: ", df.shape[0])

Rows Before:  1067371
Rows After:  1033036


## 5. Create New Dataframe for Null Customer ID and Remove from df

In [26]:
# Copy rows with missing Customer ID to a separate dataframe
missing_customerid_df = df[df["Customer ID"].isna()].copy()
missing_customerid_df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom


In [27]:
print(missing_customerid_df.shape[0])

235151


In [28]:
# Remove the rows
print("Rows Before: ", df.shape[0])  #dupes removed lowered the number from the count above in Step 3

df = df.dropna(subset=["Customer ID"]).copy()

print("Rows After: ", df.shape[0])

Rows Before:  235151
Rows After:  235151


## 6. Create New Dataframe for Invoice beginning with 'C' and Remove from df

In [30]:
# Letter C in the Invoice indicates it's a cancellation
cancellations_df = df[df["Invoice"].astype(str).str.startswith("C")].copy()
cancellations_df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia


In [31]:
print(cancellations_df.shape[0])

18390


In [33]:
print("Rows Before: ", df.shape[0])  

df = df[~df["Invoice"].astype(str).str.startswith('C')].copy()

print("Rows After: ", df.shape[0])

Rows Before:  18390
Rows After:  779495


## 7. Create New Dataframe for Quantity < 0 and Remove from df (if needed)

In [34]:
negative_qty_df = df[df["Quantity"] < 0].copy()
negative_qty_df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country


In [35]:
# 1. Check how many negative Quantity rows exist in df right now
(df["Quantity"] < 0).sum()

np.int64(0)

In [38]:
# 2. Check the dtype - make sure Quantity is actually numeric
df['Quantity'].dtype

dtype('int64')

In [39]:
# 3. Sanity check the min value
df["Quantity"].min()

np.int64(1)

## 8. Create New Dataframe for StockCode not a true StockCode and Remove from df

In [41]:
pattern = "POST|D|M|BANK CHARGES|DOT"
stockcode_msg_df = df[df["StockCode"].str.contains(pattern, case=False, na=False)]
stockcode_msg_df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
89,489439,POST,POSTAGE,3,2009-12-01 09:28:00,18.0,12682.0,France
126,489444,POST,POSTAGE,1,2009-12-01 09:55:00,141.0,12636.0,USA
173,489447,POST,POSTAGE,1,2009-12-01 10:10:00,130.0,12362.0,Belgium


In [42]:
print("Rows Before: ", df.shape[0])  

pattern = "POST|D|M|BANK CHARGES|DOT"
df = df[~df["StockCode"].str.contains(pattern, case=False, na=False)].copy()

print("Rows After: ", df.shape[0])

Rows Before:  779495
Rows After:  769953


## 8. Create New Dataframe for Price = 0 and Remove from df (could inflate Frequency)

In [46]:
price_zero_df = df[df["Price"] == 0].copy()
(df["Price"] == 0).sum()

np.int64(61)

In [47]:
price_zero_df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4607,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
6702,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
18531,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom


In [49]:
print("Rows Before: ", df.shape[0])  

df = df[~(df["Price"] == 0)].copy()

print("Rows After: ", df.shape[0])

Rows Before:  769953
Rows After:  769892


## 9. Confirm no dates outside the expected 2009-2011 range

In [50]:
print("Min Date: ", df["InvoiceDate"].min())
print("Max Date: ", df["InvoiceDate"].max())

Min Date:  2009-12-01 07:45:00
Max Date:  2011-12-09 12:50:00


## 10. Check for extreme Quantity or UnitPrice outliers

In [51]:
print("Min Qty: ", df["Quantity"].min())
print("Max Qty: ", df["Quantity"].max())

Min Qty:  1
Max Qty:  80995


In [52]:
print("Min Price: ", df["Price"].min())
print("Max Price: ", df["Price"].max())

Min Price:  0.03
Max Price:  649.5


In [68]:
df.sort_values("Quantity", ascending=False).head(15)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1031554,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
557400,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
89849,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark
125791,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
125789,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
125792,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark
125790,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark
133513,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom
133515,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom
133516,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom


In [65]:
print(cancellations_df[cancellations_df["Customer ID"].isin([16446.0, 12346.0])])

         Invoice StockCode                          Description  Quantity  \
70304    C495800    ADJUST  Adjustment by john on 26/01/2010 17        -1   
252571   C514024         M                               Manual        -1   
252572   C514024         M                               Manual        -1   
252573   C514024         M                               Manual        -1   
252574   C514024         M                               Manual        -1   
252575   C514024         M                               Manual        -3   
252576   C514024         M                               Manual        -1   
252577   C514024         M                               Manual        -4   
252578   C514024         M                               Manual        -1   
252579   C514024         M                               Manual        -1   
365675   C525099         D                             Discount        -1   
367399   C525274   TEST001              This is a test product.        -1   

## 11. Remove Orphaned Invoice Rows that have a Matching Cancellation

In [73]:
# Cancellation invoices often have the same digits as the original, prefixed with "C"
# df["Invoice"] = df["Invoice"].astype(str)
# cancellations_df['Invoice'] = cancellations_df['Invoice'].astype(str)

#cancelled_originals = cancellations_df["Invoice"].str.replace("C", "", regex=False)

orphaned_df = df[
    ((df['Customer ID'] == 16446.0) & (df['Quantity'] == 80995)) |
    ((df['Customer ID'] == 12346.0) & (df['Quantity'] == 74215))
].copy()

print("Rows Before: ", df.shape[0])

df = df.drop(orphaned_df.index).copy()

print("Rows After: ", df.shape[0])

Rows Before:  769892
Rows After:  769890


In [74]:
df.sort_values("Quantity", ascending=False).head(15)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
89849,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark
125791,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
125789,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
125792,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark
125790,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark
133516,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom
133513,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom
133514,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom
133515,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom
92646,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,2010-02-17 10:51:00,0.30,13902.0,Denmark


## 12. Create TotalPrice column

In [75]:
df["TotalPrice"] = df["Quantity"] * df["Price"]
df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0


## 13. Create Transaction Recency

In [76]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

recency_df = df.groupby("Customer ID")["InvoiceDate"].max().reset_index()
recency_df["Recency"] = (snapshot_date - recency_df["InvoiceDate"]).dt.days

recency_df.head(3)

,Customer ID,InvoiceDate,Recency
0,12346.0,2010-06-28 13:53:00,529
1,12347.0,2011-12-07 15:52:00,2
2,12348.0,2011-09-25 13:13:00,75


In [81]:
df[df["Customer ID"] == 12346.0].sort_values("InvoiceDate", ascending=False)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
250037,513774,22660,DOORMAT I LOVE LONDON,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250038,513774,22687,DOORMAT CHRISTMAS VILLAGE,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250052,513774,20685,DOORMAT RED SPOT,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250051,513774,22366,DOORMAT AIRMAIL,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250050,513774,48187,DOORMAT NEW ENGLAND,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250049,513774,48188,DOORMAT WELCOME PUPPIES,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250048,513774,22688,DOORMAT PEACE ON EARTH BLUE,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250047,513774,48185,DOORMAT FAIRY CAKE,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250046,513774,22365,DOORMAT RESPECTABLE HOUSE,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49
250045,513774,22689,DOORMAT MERRY CHRISTMAS RED,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,7.49


In [82]:
print(snapshot_date)

2011-12-10 12:50:00


## 14. Export to csv files

In [83]:
df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/cleaned_online_retail.csv', index=False)

In [84]:
duplicates_df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/duplicate_rows.csv', index=False)

In [85]:
missing_customerid_df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/missing_customerid.csv', index=False)

In [86]:
cancellations_df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/cancelled_orders.csv', index=False)

In [87]:
negative_qty_df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/negative_qty.csv', index=False)

In [88]:
stockcode_msg_df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/invalid_stockcode.csv', index=False)

In [89]:
price_zero_df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/price_is_zero.csv', index=False)

In [90]:
orphaned_df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/orphaned.csv', index=False)

In [91]:
recency_df.to_csv('C:/Projects/online-retail-rfm-segmentation/data/recency.csv', index=False)